# OECD Client Demonstration

Ce notebook illustre les différentes fonctionnalités du client OECD pour télécharger des données via l'API SDMX.

## Table des matières

1. [Setup](#section-0)
2. [Lister tous les dataflows](#section-1)
3. [Obtenir la structure d'un dataflow](#section-2)
4. [Requête basique avec positions](#section-3)
5. [Requête avec noms de dimensions](#section-4)
6. [Split dimensions](#section-5)
7. [Objet QueryRequest](#section-6)
8. [Filtrage par date de mise à jour](#section-7)
9. [Chargement depuis la configuration](#section-8)
10. [Gestion des erreurs](#section-9)
11. [Conseils de performance](#section-10)

## Section 0: Setup <a id="section-0"></a>

Importation des modules nécessaires et configuration de l'environnement.

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

import sys
import yaml
import pandas as pd
from pathlib import Path

# Ajout du répertoire parent au path
sys.path.append('..')

from macroforecast.datasets.oecd import OECDClient, QueryRequest

# Configuration de l'affichage pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("Setup complete!")

## Section 1: Lister tous les dataflows <a id="section-1"></a>

Utilisation de `list_all_dataflows()` pour énumérer tous les dataflows disponibles.

In [ ]:
# Initialisation du client
client = OECDClient()

# Récupération de tous les dataflows
dataflows = client.list_all_dataflows()

print(f"Nombre total de dataflows: {len(dataflows)}")
print("\nPremiers dataflows:")
dataflows.head(10)

## Section 2: Obtenir la structure d'un dataflow <a id="section-2"></a>

Démonstration de `get_structure()` pour comprendre les dimensions d'un dataflow.

In [ ]:
# Extraction de la structure du dataflow KEI
structure = client.get_structure(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="4.0"
)

print(f"Agency: {structure.agency}")
print(f"Dataflow: {structure.dataflow}")
print(f"Nombre de dimensions: {structure.num_dimensions}")
print("\nDimensions:")
for dim in structure.dimensions:
    print(f"  Position {dim.position}: {dim.name} - {dim.description}")

## Section 3: Requête basique avec positions <a id="section-3"></a>

Utilisation de `get_data()` avec dimensions par position (0, 1, 2...).

In [ ]:
# Requête avec positions numériques
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        0: ["FRA"],  # Position 0 = REF_AREA
        1: ["M"],    # Position 1 = FREQ
        2: ["LI"],   # Position 2 = MEASURE
    },
    start_period="2020"
)

print(f"Nombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

## Section 4: Requête avec noms de dimensions <a id="section-4"></a>

Utilisation de `get_data()` avec dimensions par nom (plus lisible et robuste).

In [ ]:
# Requête avec noms de dimensions
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA", "DEU"],  # France et Allemagne
        "FREQ": "M",                  # Fréquence mensuelle
        "MEASURE": "LI",              # Leading Indicator
    },
    start_period="2020"
)

print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['REF_AREA'].unique())}")
df.head()

## Section 5: Split dimensions <a id="section-5"></a>

Démonstration du paramètre `split_dimensions` pour gérer les requêtes volumineuses.

In [ ]:
# Requête avec split_dimensions
# Au lieu d'une seule requête pour 7 pays, on génère 7 requêtes séparées
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["CAN", "FRA", "DEU", "ITA", "JPN", "GBR", "USA"],
        "FREQ": "M",
        "MEASURE": "LI",
    },
    start_period="2020",
    split_dimensions=["REF_AREA"]  # Générera 7 requêtes séparées
)

print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['REF_AREA'].unique())}")
print("\nStatistiques par pays:")
print(df.groupby('REF_AREA').size())

## Section 6: Objet QueryRequest <a id="section-6"></a>

Création et exécution d'une `QueryRequest` pour une manipulation plus flexible.

In [ ]:
# Création d'une QueryRequest
query = QueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    version="+",
    dimensions={
        "REF_AREA": ["FRA"],
        "FREQ": "M",
        "MEASURE": "LI"
    },
    start_period="2020"
)

print(f"Dataflow key: {query.get_dataflow_key()}")
print(f"Dimensions: {query.dimensions}")

# Exécution de la requête
df = client.execute_query(query)

print(f"\nNombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

## Section 7: Filtrage par date de mise à jour <a id="section-7"></a>

Démonstration de `filter_updated_queries()` pour éviter de télécharger des données non mises à jour.

In [ ]:
# Création de plusieurs requêtes
queries = [
    QueryRequest(
        agency="OECD.SDD.STES",
        dataflow="DSD_KEI@DF_KEI",
        dimensions={"REF_AREA": ["FRA"], "FREQ": "M", "MEASURE": "LI"},
        start_period="2020"
    ),
    QueryRequest(
        agency="OECD.SDD.STES",
        dataflow="DSD_KEI@DF_KEI",
        dimensions={"REF_AREA": ["DEU"], "FREQ": "M", "MEASURE": "LI"},
        start_period="2020"
    ),
]

print(f"Nombre de requêtes totales: {len(queries)}")

# Test 1: Filtrage avec date ancienne (devrait tout garder)
print("\n--- Test avec date ancienne (2020-01-01) ---")
updated_queries_old = client.filter_updated_queries(
    queries,
    updated_since="2020-01-01"
)
print(f"Requêtes filtrées: {len(updated_queries_old)}/{len(queries)}")

# Test 2: Filtrage avec date récente
print("\n--- Test avec date récente (2024-01-01) ---")
updated_queries_recent = client.filter_updated_queries(
    queries,
    updated_since="2024-01-01"
)
print(f"Requêtes filtrées: {len(updated_queries_recent)}/{len(queries)}")

# Test 3: Aucun filtrage avec None
print("\n--- Test sans filtrage (updated_since=None) ---")
all_queries = client.filter_updated_queries(
    queries,
    updated_since=None
)
print(f"Requêtes retournées: {len(all_queries)}/{len(queries)}")

## Section 8: Chargement depuis la configuration <a id="section-8"></a>

Chargement des `query_requests` depuis le fichier de configuration YAML.

In [ ]:
# Chargement de la configuration
config_path = Path('../config/datasets/oecd.yaml')
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("Requêtes configurées:")
if 'query_requests' in config:
    for query_name in config['query_requests'].keys():
        print(f"  - {query_name}")

    # Exemple: chargement et exécution de la première requête
    query_cfg = config['query_requests']['kei_g7_monthly']

    print(f"\nConfiguration de 'kei_g7_monthly':")
    print(f"  Agency: {query_cfg['agency']}")
    print(f"  Dataflow: {query_cfg['dataflow']}")
    print(f"  Dimensions: {list(query_cfg['dimensions'].keys())}")

    # Création de la QueryRequest depuis la config
    query = QueryRequest(
        agency=query_cfg['agency'],
        dataflow=query_cfg['dataflow'],
        version=query_cfg.get('version', '+'),
        dimensions=query_cfg['dimensions'],
        start_period=query_cfg.get('start_period'),
        end_period=query_cfg.get('end_period'),
        split_dimensions=query_cfg.get('split_dimensions')
    )

    print(f"\nQueryRequest créée: {query.get_dataflow_key()}")
    print("\nPour exécuter la requête, décommentez la ligne suivante:")
    print("# df = client.execute_query(query)")

else:
    print("⚠ Section 'query_requests' non trouvée dans la configuration")

## Section 9: Gestion des erreurs <a id="section-9"></a>

Démonstration de la gestion des erreurs avec des requêtes invalides.

In [ ]:
# Test 1: Agency invalide
print("Test 1: Agency invalide")
try:
    df = client.get_data(
        agency="INVALID_AGENCY",
        dataflow="INVALID_DATAFLOW",
        dimensions={"REF_AREA": ["FRA"]}
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}")

# Test 2: Dimension invalide
print("\nTest 2: Dimension invalide")
try:
    df = client.get_data(
        agency="OECD.SDD.STES",
        dataflow="DSD_KEI@DF_KEI",
        dimensions={"INVALID_DIM": ["VALUE"]}
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:100]}")

print("\n✓ Tests de gestion d'erreurs terminés")

## Section 10: Conseils de performance <a id="section-10"></a>

Meilleures pratiques pour optimiser les téléchargements de données OECD.

### 1. Rate Limiting
- Le client OECD implémente automatiquement un rate limiting de **60 requêtes par heure**
- Aucune action requise de votre part, c'est géré automatiquement

### 2. Split Dimensions
- Utilisez `split_dimensions` pour les requêtes avec beaucoup de valeurs
- Génère plusieurs petites requêtes au lieu d'une grosse requête
- Exemple: 7 pays × 14 transactions = 98 requêtes avec `split_dimensions=["REF_AREA", "TRANSACTION"]`

```python
# Bon: split en plusieurs requêtes
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["CAN", "FRA", "DEU", "ITA", "JPN", "GBR", "USA"]},
    split_dimensions=["REF_AREA"]
)
```

### 3. Filtrage par date
- Utilisez `filter_updated_queries()` pour éviter les téléchargements inutiles
- Ne télécharge que les dataflows mis à jour depuis une date donnée
- Passez `updated_since=None` pour ne pas filtrer

```python
# Ne télécharge que les données mises à jour cette semaine
updated_queries = client.filter_updated_queries(
    queries,
    updated_since="2024-12-15"
)
```

### 4. Formats de réponse
- **CSV_LABELS** (défaut): CSV avec labels lisibles - recommandé
- **CSV**: CSV avec codes - plus léger mais moins lisible
- **JSON**: Format JSON - plus flexible mais plus lourd

```python
from macroforecast.datasets.sdmx import ResponseFormat

query = QueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    format=ResponseFormat.CSV  # Plus rapide si vous connaissez les codes
)
```

### 5. Gestion des doublons
- Par défaut: `on_duplicate="warn"` (affiche un warning)
- Options: `"ignore"`, `"warn"`, `"raise"`

```python
query = QueryRequest(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    on_duplicate="raise"  # Lève une exception en cas de doublon
)
```

### 6. Limitation temporelle
- Utilisez `start_period` et `end_period` pour limiter la période
- Ou `last_n_observations` pour ne récupérer que les N dernières observations

```python
# Seulement les 12 derniers mois
df = client.get_data(
    agency="OECD.SDD.STES",
    dataflow="DSD_KEI@DF_KEI",
    dimensions={"REF_AREA": ["FRA"]},
    last_n_observations=12
)
```

## Conclusion

Ce notebook a démontré toutes les fonctionnalités principales du client OECD:

✓ Énumération des dataflows disponibles  
✓ Récupération de la structure des dataflows  
✓ Requêtes avec positions et noms de dimensions  
✓ Gestion des requêtes volumineuses avec split_dimensions  
✓ Utilisation de QueryRequest pour plus de flexibilité  
✓ Filtrage intelligent par date de mise à jour  
✓ Chargement depuis fichier de configuration  
✓ Gestion robuste des erreurs  
✓ Bonnes pratiques de performance  

Pour plus d'informations, consultez:
- Le fichier `macroforecast/datasets/oecd.py`
- La configuration `config/datasets/oecd.yaml`
- Le script `scripts/download_oecd_data.py`